In [8]:
import numpy as np
import matplotlib.pyplot as plt
import ripser
import persim
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist, squareform
import gudhi as gd
from gudhi.representations import Landscape, Silhouette, PersistenceImage
import tadasets
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import networkx as nx
import nibabel as nib
import os
from tqdm import tqdm
from skimage.transform import resize

target_shape = (128, 128, 64)

In [131]:
def compute_persistent_homology(data, max_dim=1, max_radius=1000.0):
    """
    Улучшенная версия вычисления персистентных гомологий для обнаружения большего числа особенностей.
    """
    
    data_reduced = data
    
    # Если входные данные слишком маленькие, добавляем небольшой шум
    if data_reduced.shape[0] == 1:
        noisy_copies = []
        for i in range(9):  
            noise = np.random.normal(0, 0.01, data_reduced.shape)
            noisy_copies.append(data_reduced + noise)
        
        data_reduced = np.vstack([data_reduced] + noisy_copies)
    
    sample_size = min(50, data_reduced.shape[0])
    indices = np.random.choice(data_reduced.shape[0], sample_size, replace=False)
    data_sample = data_reduced[indices]
    
    try:
        metrics = ['euclidean', 'correlation', 'cosine']
        
        for metric in metrics:
            
            dist_matrix = squareform(pdist(data_sample, metric=metric))
            
            if np.count_nonzero(dist_matrix) < dist_matrix.size / 2:
                dist_matrix += np.random.uniform(0, 0.01, dist_matrix.shape)
                np.fill_diagonal(dist_matrix, 0)  
            
            diagrams = ripser.ripser(dist_matrix, maxdim=max_dim, distance_matrix=True, thresh=max_radius)['dgms']
            
            if len(diagrams) > 1 and len(diagrams[1]) > 0:
                return diagrams
        
        dist_matrix = squareform(pdist(data_sample, metric='euclidean'))
        
        if np.max(dist_matrix) > 0:
            dist_matrix = dist_matrix / np.max(dist_matrix) * max_radius
        
        diagrams = ripser.ripser(dist_matrix, maxdim=max_dim, distance_matrix=True, 
                               thresh=max_radius * 2)['dgms']
        
        return diagrams
    except Exception as e:
        print(f"Ошибка при вычислении персистентных гомологий: {e}")
        return [np.array([]) for _ in range(max_dim + 1)]


In [132]:

def create_artificial_anomalies(mri_data, anomaly_ratio=0.2, anomaly_intensity=3.0):
    """
    Создает искусственные аномалии в МРТ данных.
    
    Args:
        mri_data: Исходные МРТ данные
        anomaly_ratio: Доля данных с аномалиями
        anomaly_intensity: Интенсивность аномалий
        
    Returns:
        Данные с аномалиями и список индексов аномальных образцов
    """
    mri_data_anomaly = mri_data.copy()
    num_samples = mri_data.shape[0]
    num_anomalies = int(num_samples * anomaly_ratio)
    
    # Выбираем случайные образцы для внесения аномалий
    anomaly_indices = np.random.choice(num_samples, num_anomalies, replace=False)
    
    for idx in anomaly_indices:
        sample_3d = mri_data[idx].reshape(target_shape)
        
        # Создаем случайную локальную аномалию
        x_center = np.random.randint(20, target_shape[0]-20)
        y_center = np.random.randint(20, target_shape[1]-20)
        z_center = np.random.randint(10, target_shape[2]-10)
        
        radius = np.random.randint(20, 30)
        
        # Создаем шар аномалии
        x, y, z = np.indices((target_shape))
        dist = np.sqrt((x - x_center)**2 + (y - y_center)**2 + (z - z_center)**2)
        anomaly_mask = dist <= radius
        
        if np.random.rand() > 0.5:
            sample_3d[anomaly_mask] *= anomaly_intensity
        else:
            sample_3d[anomaly_mask] /= anomaly_intensity
        
        mri_data_anomaly[idx] = sample_3d.flatten()
    
    return mri_data_anomaly, anomaly_indices


In [136]:
def evaluate_anomaly_detection(mri_data, anomaly_ratio=0.2):
    """
    Исправленная функция оценки качества обнаружения аномалий.
    """

    mri_data_with_anomalies, anomaly_indices = create_artificial_anomalies(mri_data, anomaly_ratio, anomaly_intensity=100.0)
    
    print(f"Создано {len(anomaly_indices)} аномальных МРТ образцов из {len(mri_data)} всего")
    
    all_diagrams = []
    all_topological_scores = []
    
    for i in tqdm(range(len(mri_data_with_anomalies)), desc="Анализ образцов"):
        sample = mri_data_with_anomalies[i].reshape(1, -1)
        
        try:
            diagrams = compute_persistent_homology(sample, max_dim=1, max_radius=100.0)
            anomaly_score = 0
            
            if len(diagrams) > 1 and len(diagrams[1]) > 0:
                h1_diagram = diagrams[1]
                
                persistences = []
                for j in range(len(h1_diagram)):
                    if isinstance(h1_diagram[j], np.ndarray) and len(h1_diagram[j]) == 2:
                        birth, death = h1_diagram[j]
                        if not np.isinf(death):
                            persistences.append(death - birth)
                
                if persistences:
                    anomaly_score = np.mean(persistences)
            else:
                h0_diagram = diagrams[0]
                
                finite_persistences = []
                for j in range(len(h0_diagram)):
                    if isinstance(h0_diagram[j], np.ndarray) and len(h0_diagram[j]) == 2:
                        birth, death = h0_diagram[j]
                        if not np.isinf(death):
                            finite_persistences.append(death - birth)
                
                if finite_persistences:
                    anomaly_score = np.std(finite_persistences)
                else:
                    anomaly_score = len(h0_diagram) * 0.01
            
            all_diagrams.append(diagrams)
            all_topological_scores.append(anomaly_score)
        except Exception as e:
            print(f"Ошибка при анализе образца {i}: {e}")
            all_diagrams.append(None)
            all_topological_scores.append(0)
    
    all_topological_scores = np.array(all_topological_scores)
    
    true_labels = np.zeros(len(mri_data_with_anomalies))
    true_labels[anomaly_indices] = 1
    
    if np.std(all_topological_scores) < 1e-10:
        print("ВНИМАНИЕ: Все оценки аномальности почти одинаковые. Метрики могут быть ненадежными.")
        all_topological_scores += np.random.normal(0, 1e-5, all_topological_scores.shape)
    
    if np.max(all_topological_scores) > np.min(all_topological_scores):
        normalized_scores = (all_topological_scores - np.min(all_topological_scores)) / (np.max(all_topological_scores) - np.min(all_topological_scores))
    else:
        normalized_scores = np.zeros_like(all_topological_scores)

    from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
    
    try:
        roc_auc = roc_auc_score(true_labels, normalized_scores)
        precision, recall, _ = precision_recall_curve(true_labels, normalized_scores)
        pr_auc = auc(recall, precision)
        
        print(f"\nРезультаты обнаружения аномалий:")
        print(f"ROC AUC: {roc_auc:.4f}")
        print(f"PR AUC: {pr_auc:.4f}")
        
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        from sklearn.metrics import roc_curve
        fpr, tpr, _ = roc_curve(true_labels, normalized_scores)
        plt.plot(fpr, tpr)
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve (AUC = {roc_auc:.4f})')
        
        plt.subplot(1, 3, 2)
        plt.plot(recall, precision)
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'PR Curve (AUC = {pr_auc:.4f})')
        
        plt.subplot(1, 3, 3)
        plt.hist(normalized_scores[true_labels == 0], alpha=0.5, bins=20, label='Нормальные')
        plt.hist(normalized_scores[true_labels == 1], alpha=0.5, bins=20, label='Аномальные')
        plt.xlabel('Топологическая оценка аномальности')
        plt.ylabel('Количество образцов')
        plt.legend()
        plt.title('Распределение оценок')
        
        plt.tight_layout()
        plt.savefig('anomaly_detection_evaluation.png')
        plt.close()
        
        threshold = 0.2
        predicted_anomalies = normalized_scores >= threshold
        
        tp_indices = np.where((predicted_anomalies == 1) & (true_labels == 1))[0]
        fn_indices = np.where((predicted_anomalies == 0) & (true_labels == 1))[0]
        fp_indices = np.where((predicted_anomalies == 1) & (true_labels == 0))[0]
        
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        
        def plot_example(ax, idx, title):
            sample_3d = mri_data_with_anomalies[idx].reshape(target_shape)
            ax.imshow(sample_3d[:, :, sample_3d.shape[2]//2], cmap='gray')
            ax.set_title(f"{title}\nScore: {normalized_scores[idx]:.4f}")
            ax.axis('off')
        
        # Примеры TP
        for i in range(min(3, len(tp_indices))):
            if i < len(tp_indices):
                plot_example(axes[0, i], tp_indices[i], f"TP {i+1}")
        
        # Примеры FN
        for i in range(min(3, len(fn_indices))):
            if i < len(fn_indices):
                plot_example(axes[1, i], fn_indices[i], f"FN {i+1}")
        
        # Примеры FP
        for i in range(min(3, len(fp_indices))):
            if i < len(fp_indices):
                plot_example(axes[2, i], fp_indices[i], f"FP {i+1}")
        
        plt.tight_layout()
        plt.savefig('anomaly_examples.png')
        plt.close()
        
        return roc_auc, pr_auc, normalized_scores, true_labels
        
    except Exception as e:
        print(f"Ошибка при расчете метрик: {e}")
        return 0, 0, normalized_scores, true_labels


In [ ]:
data_dir = './data/'
target_shape = (128, 128, 64)

def load_mri_images(data_dir):
    images = []
    file_list = [f for f in os.listdir(data_dir) if f.endswith('.nii') or f.endswith('.nii.gz')]
    
    for file_name in tqdm(file_list, desc="Загрузка MRI-изображений"):
        img_path = os.path.join(data_dir, file_name)
        img = nib.load(img_path)
        img_data = img.get_fdata()

        img_resized = resize(img_data, target_shape, anti_aliasing=True)

        img_vector = img_resized.flatten()
        images.append(img_vector)
    
    return np.array(images)

mri_data = load_mri_images(data_dir)


Загрузка MRI-изображений: 100%|██████████| 582/582 [01:59<00:00,  4.85it/s]


In [140]:
# Оценка качества определения аномалий
roc_auc, pr_auc, scores, true_labels = evaluate_anomaly_detection(mri_data[:100], anomaly_ratio=0.8)

print(f"\nИтоговые результаты обнаружения аномалий:")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"PR AUC: {pr_auc:.4f}")

Создано 80 аномальных МРТ образцов из 100 всего


Анализ образцов: 100%|██████████| 100/100 [00:29<00:00,  3.36it/s]



Результаты обнаружения аномалий:
ROC AUC: 0.5672
PR AUC: 0.8250

Итоговые результаты обнаружения аномалий:
ROC AUC: 0.5672
PR AUC: 0.8250
